## Planning Design

#### Import libraries

In [ ]:
from agents import Agent, Runner, trace
import os, asyncio
from dotenv import load_dotenv

load_dotenv(override=True)

#### Define Agents

In [ ]:
# Agent 1: Planner
planner_agent = Agent(
    name="BioPlanner",
    instructions=(
        "Given a bioinformatics task, break it down into a step-by-step plan, 3 steps max"
        "Be specific and concise. Only output the plan."
    ),
    model="gpt-4.1-nano"
)

# Agent 2: Executor (for demo, just echoes the step)
executor_agent = Agent(
    name="BioExecutor",
    instructions=(
        "Given a step from a bioinformatics plan, explain how you would execute it or what the expected result is in 15 words or less."
    ),
    model="gpt-4.1-nano"
)


#### Define the workflow

In [ ]:

async def planning_pipeline(task):
    # Step 1: Generate a plan
    plan_result = await Runner.run(planner_agent, task)
    plan = plan_result.final_output
    print("Generated Plan:\n", plan)

    # Step 2: Execute each step
    steps = [line for line in plan.split('\n') if line.strip() and line[0].isdigit()]
    for step in steps:
        await asyncio.sleep(20)  # To avoid rate limits
        exec_result = await Runner.run(executor_agent, step)
        print(f"Execution of '{step}':\n{exec_result.final_output}\n")

#### Execute the workflow

In [ ]:
# Example usage
with trace("planning_pipeline"):
    task = "Analyze a DNA sequence for mutations and annotate their clinical significance."
    await planning_pipeline(task)